<div style="background:#0B3C49;color:white;padding:14px 18px;border-radius:8px">
<b style="color:#F2A541">INTERNATIONAL BALKAN UNIVERSITY · DS 501 INTRODUCTION TO DATA SCIENCE</b><br>
<span style="font-size:1.4em">Lab 3 · Acquiring Data: Files, APIs, Web Pages, Tidy Data</span><br>
Week 3 · Lessons 3-1 to 3-3 · about 75 minutes
</div>

## Learning goals
1. Read data from files and URLs with pandas.
2. Request data from a **REST API** and turn nested **JSON** into a table.
3. Extract a table from an **HTML** page (web scraping) responsibly.
4. Reshape data into **tidy** form (Wickham, 2014) and write a short **data dictionary** (metadata).

> **Note on connectivity.** Sections 2–4 call public web services. If you are offline or a service is down, each cell falls back to a small built-in sample so the notebook still runs. Fallback values are **synthetic** and only show the structure; always say in your work which source you actually used.

In [ ]:
import json, io
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import requests
pd.set_option("display.max_columns", 30)

## 1. Reading files from a URL
`pd.read_csv` accepts a URL directly. Always check the shape, types and a few rows after loading.

In [ ]:
tips = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
print(tips.shape)
print(tips.dtypes)
tips.head(3)

**✏️ Your turn.** Save `tips` as an Excel file (`tips.xlsx`) and as JSON lines (`tips.jsonl`, one record per line), read both back, and confirm the shapes match.

In [ ]:
# Hint: to_excel / read_excel need openpyxl (pre-installed in Colab); JSON lines uses lines=True
# YOUR CODE HERE

## 2. A REST API: daily weather in Skopje (Open-Meteo)
[Open-Meteo](https://open-meteo.com) is free and needs no API key. An API request is a URL with **parameters**; the response is **JSON**.

In [ ]:
URL = "https://archive-api.open-meteo.com/v1/archive"
params = {"latitude": 41.9981, "longitude": 21.4254, "start_date": "2024-01-01", "end_date": "2024-12-31",
          "daily": "temperature_2m_mean,precipitation_sum", "timezone": "Europe/Skopje"}
try:
    r = requests.get(URL, params=params, timeout=20)
    r.raise_for_status()
    weather_json = r.json()
    SOURCE = "Open-Meteo archive API (live)"
except Exception as e:
    print("API not reachable, using SYNTHETIC fallback:", type(e).__name__)
    days = pd.date_range("2024-01-01", "2024-12-31", freq="D")
    rng = np.random.default_rng(0)
    temp = 13 + 12 * np.sin(2 * np.pi * (days.dayofyear - 105) / 366) + rng.normal(0, 2.5, len(days))
    rain = np.where(rng.random(len(days)) < 0.25, rng.gamma(2, 3, len(days)), 0)
    weather_json = {"latitude": 41.99, "longitude": 21.43, "daily_units": {"temperature_2m_mean": "°C", "precipitation_sum": "mm"},
                    "daily": {"time": days.strftime("%Y-%m-%d").tolist(), "temperature_2m_mean": temp.round(1).tolist(),
                              "precipitation_sum": rain.round(1).tolist()}}
    SOURCE = "SYNTHETIC fallback (structure only)"
print("Source:", SOURCE)
print(json.dumps({k: (v if k != "daily" else {kk: vv[:3] for kk, vv in v.items()}) for k, v in weather_json.items()}, indent=1)[:600])

In [ ]:
weather = pd.DataFrame(weather_json["daily"])
weather["time"] = pd.to_datetime(weather["time"])
weather = weather.set_index("time")
weather.head()

**✏️ Your turn.** Compute the **monthly** mean temperature and total precipitation, and plot the monthly mean temperature as a line chart.

In [ ]:
# Hint: weather.resample("MS").agg({...})
# YOUR CODE HERE

**✏️ Your turn (written).** Look at the request URL and parameters. Which parameters would you change to get hourly data for Bitola for the last five years? What could go wrong with such a large request?

*Your answer:* 

## 3. Nested JSON: World Bank indicators
The World Bank API returns a list `[metadata, records]`, and each record has nested objects. `pd.json_normalize` flattens them.

In [ ]:
WB = "https://api.worldbank.org/v2/country/MKD;ALB;SRB/indicator/NY.GDP.PCAP.CD"
try:
    r = requests.get(WB, params={"format": "json", "per_page": 300, "date": "2010:2023"}, timeout=20)
    r.raise_for_status()
    wb_json = r.json()
    WB_SOURCE = "World Bank API (live)"
except Exception as e:
    print("API not reachable, using SYNTHETIC fallback:", type(e).__name__)
    recs = []
    for code, name, base in [("MKD", "North Macedonia", 4500), ("ALB", "Albania", 4000), ("SRB", "Serbia", 5500)]:
        for i, yr in enumerate(range(2010, 2024)):
            recs.append({"indicator": {"id": "NY.GDP.PCAP.CD", "value": "GDP per capita (current US$)"},
                         "country": {"id": code[:2], "value": name}, "countryiso3code": code,
                         "date": str(yr), "value": round(base * (1.04 ** i), 1)})
    wb_json = [{"page": 1, "total": len(recs)}, recs]
    WB_SOURCE = "SYNTHETIC fallback (structure only)"
print("Source:", WB_SOURCE)
print(json.dumps(wb_json[1][0], indent=1))

**✏️ Your turn.** Flatten `wb_json[1]` with `pd.json_normalize`, keep `country.value`, `date`, `value`, convert `date` to integer, and draw one line per country.

In [ ]:
# Hint: pd.json_normalize(wb_json[1]) creates columns like 'country.value'
# YOUR CODE HERE

## 4. Web scraping an HTML table
Scraping extracts data from web pages. **Check the site's terms and `robots.txt`, identify yourself, and do not overload servers.** Below we parse a small HTML page; `pd.read_html` and BeautifulSoup do the same on live pages.

In [ ]:
html = '''
<html><body>
<h2>Example: public transport lines (fictional)</h2>
<table id="lines">
 <tr><th>Line</th><th>From</th><th>To</th><th>Length (km)</th><th>Buses per hour</th></tr>
 <tr><td>2</td><td>Gjorce Petrov</td><td>Kisela Voda</td><td>14.2</td><td>6</td></tr>
 <tr><td>5</td><td>Karpos</td><td>Aerodrom</td><td>11.8</td><td>8</td></tr>
 <tr><td>22</td><td>Center</td><td>Butel</td><td>7.5</td><td>4</td></tr>
 <tr><td>57</td><td>Center</td><td>Saraj</td><td>18.9</td><td>2</td></tr>
</table>
<p class="note">Updated: 2026-09-01</p>
</body></html>'''
lines = pd.read_html(io.StringIO(html))[0]
lines

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
rows = [[td.get_text(strip=True) for td in tr.find_all(["td", "th"])] for tr in soup.select("#lines tr")]
print(rows[:2])
print("Note text:", soup.find("p", class_="note").get_text())

**✏️ Your turn.** Using the BeautifulSoup `rows` list, build a DataFrame with correct numeric types and add a column `km_per_bus` = length / buses per hour. Which line has the longest distance per bus?

In [ ]:
# Hint: the first row is the header; convert columns with astype
# YOUR CODE HERE

In [ ]:
# Optional live example (needs internet; wrapped so the notebook still runs offline)
try:
    resp = requests.get("https://en.wikipedia.org/wiki/List_of_cities_in_North_Macedonia",
                        headers={"User-Agent": "DS501-student-lab (educational use)"}, timeout=20)
    tables = pd.read_html(io.StringIO(resp.text))
    print(len(tables), "tables found; first one:")
    display(tables[0].head())
except Exception as e:
    print("Live scraping skipped:", type(e).__name__)

**✏️ Your turn (written).** Name two legal or ethical limits on web scraping and one technical reason scrapers break.

*Your answer:* 

## 5. Tidy data
Wickham (2014): in tidy data **each variable is a column, each observation is a row, and each type of observational unit is a table**. Spreadsheets often store data *wide*.

In [ ]:
wide = pd.DataFrame({
    "student": ["S01", "S02", "S03", "S04"],
    "programme": ["MSc DS", "MSc DS", "MBA", "MSc CS"],
    "quiz_w1": [8, 6, 9, 7], "quiz_w2": [7, 5, 10, 8], "quiz_w3": [9, None, 9, 6]})
wide

**✏️ Your turn.** Reshape `wide` into tidy (long) form with columns `student`, `programme`, `week` (integer) and `score`. Then compute each student's mean score.

In [ ]:
# Hint: DataFrame.melt(id_vars=[...], var_name=..., value_name=...)
# YOUR CODE HERE

**✏️ Your turn (written).** Why is the long format easier for analysis and plotting than the wide one? Give one situation where the wide format is better.

*Your answer:* 

## 6. Metadata: a data dictionary
Good data has documentation: what each column means, its type, unit, allowed values, and source.

**✏️ Your turn.** Create a data dictionary DataFrame for the `weather` table with columns `column`, `type`, `unit`, `description`, `source`.

In [ ]:
# Hint: a list of dicts passed to pd.DataFrame
# YOUR CODE HERE

**✏️ Your turn (written).** For your **course project**, list the data sources you plan to use, their format (CSV, JSON, API, HTML, database), and how you will access them. This is your week 3 project checkpoint.

*Your answer:* 

---
### Submission checklist
- [ ] All ✏️ cells completed
- [ ] `Kernel → Restart & Run All` runs without errors
- [ ] Notebook saved and uploaded to Moodle before the next class